# TREA on TKGL-Wikidata Cross-Lingual (memory-safe)

**TREA** (Xu, Su, Xiong, Lehmann — WWW 2022) on `tkgl_wikidata_crosslingual`.

Full TKGL can have **~1.2M EN entities**. This notebook:
1. Keeps an **alignment-centric subgraph** (`HOP_EXPANSION=0` by default)
2. Uses the **correct TRALayer** (scores from time/rel emb; Wt/Wr on features)
3. Uses **sampled MML** when |V| is large (avoids N×N distance matrices)

### Zip layout
```
tkgl_wikidata_crosslingual.zip
├── en_fa/  en_zh/  en_ja/  en_fr/
    ├── alignment_all.tsv
    ├── quadruples_en.tsv      # h r t ts  (integer time id)
    └── quadruples_<lang>.tsv
```

### Colab
1. **Restart runtime** first if you just ran TEA-GNN/STEA
2. GPU (High-RAM if available)
3. Upload the zip, set `LANG_PAIR`, run top→bottom

**Caveat:** paper reconstruction; document subgraph + sampled loss in the thesis.

### Memory-safe update (fixes `CUDA out of memory ... Tried to allocate 4.72 GiB`)
The attention layer used to build several `(num_edges, 3*DIM)` tensors per layer and keep them for
backward. It now aggregates neighbours **in edge chunks** with a custom autograd function (the output and
every gradient are numerically identical - verified against the previous implementation), uses optional
activation checkpointing, and the final evaluation is chunked too. Tune with `CHUNK_EDGES` and
`USE_CHECKPOINT` in the config cell.


In [ ]:
import os, gc, zipfile, random
from pathlib import Path
from collections import defaultdict

import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F

SEED = 200
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)
    torch.cuda.empty_cache()

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", DEVICE)
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
    print("GB:", round(torch.cuda.get_device_properties(0).total_memory/1e9, 2))


In [ ]:
from google.colab import drive
drive.mount("/content/drive")


In [ ]:
# ============================================================
# Config
# ============================================================

LANG_PAIR = "en_fr"  # en_fa | en_zh | en_ja | en_fr

ZIP_CANDIDATES = [
    Path("/content/drive/MyDrive/tkgl_wikidata_crosslingual.zip"),
    Path("/content/tkgl_wikidata_crosslingual.zip"),
    Path("/content/drive/MyDrive/en_fa.zip"),
    Path("/content/en_fa.zip"),
]
EXTRACT_ROOT = Path("/content/tkgl_wikidata_crosslingual")

# memory knobs
HOP_EXPANSION = 0
MAX_QUADS_EN = None
MAX_QUADS_OTHER = None
MAX_NEIGHBORS_PER_NODE = 30

# TREA hyperparams
TRAIN_SEED = 200
DIM = 100
DEPTH = 2
GAMMA = 0.0
LAMBDA_T = 0.001
LR = 0.005
EPOCHS = 80

# MML: if |V| > this, sample negatives instead of full matrix
FULL_MML_MAX_ENTITIES = 30_000
NUM_NEG_SAMPLES = 64

TOP_K = (1, 5, 10)
CSLS_K = 10
EVAL_MAX_PAIRS = 5000     # pairs used for the periodic eval during training

# ---- GPU-memory knobs (new) ----
CHUNK_EDGES = 262_144     # edges aggregated per chunk. Lower (65_536) => less GPU memory, a bit slower
USE_CHECKPOINT = True     # recompute each attention layer in backward => far fewer stored activations

print(LANG_PAIR, "HOP", HOP_EXPANSION, "DIM", DIM)


In [ ]:
# ============================================================
# Extract
# ============================================================
from pathlib import Path
import zipfile

# Fallback if Config cell was skipped
if "ZIP_CANDIDATES" not in globals():
    print("WARNING: Config cell not run — using default ZIP_CANDIDATES / LANG_PAIR")
    LANG_PAIR = globals().get("LANG_PAIR", "en_fa")
    ZIP_CANDIDATES = [
        Path("/content/drive/MyDrive/tkgl_wikidata_crosslingual.zip"),
        Path("/content/tkgl_wikidata_crosslingual.zip"),
        Path("/content/drive/MyDrive/en_fa.zip"),
        Path("/content/en_fa.zip"),
    ]
    EXTRACT_ROOT = Path("/content/tkgl_wikidata_crosslingual")

zip_path = next((p for p in ZIP_CANDIDATES if p.exists()), None)
if zip_path is None:
    found = list(Path("/content").rglob("*crosslingual*.zip")) + list(Path("/content").rglob("en_fa.zip"))
    zip_path = found[0] if found else None
if zip_path is None:
    raise FileNotFoundError(
        "tkgl zip not found. Run the Config cell first, or put "
        "tkgl_wikidata_crosslingual.zip in Drive or /content."
    )

print("Zip:", zip_path)
if not (EXTRACT_ROOT / ".extracted").exists():
    EXTRACT_ROOT.mkdir(parents=True, exist_ok=True)
    with zipfile.ZipFile(zip_path, "r") as zf:
        zf.extractall(EXTRACT_ROOT)
    (EXTRACT_ROOT / ".extracted").write_text("ok")

cands = [p.parent for p in EXTRACT_ROOT.rglob("alignment_all.tsv")
         if p.parent.name == LANG_PAIR or LANG_PAIR in str(p.parent)]
if not cands:
    avail = sorted({p.parent.name for p in EXTRACT_ROOT.rglob("alignment_all.tsv")})
    raise FileNotFoundError(f"{LANG_PAIR} missing. Available: {avail}")
PAIR_DIR = cands[0]
lang_a, lang_b = LANG_PAIR.split("_")
print("PAIR_DIR", PAIR_DIR)
print("langs:", lang_a, lang_b)


In [5]:
# ============================================================
# Stream load + alignment subgraph
# ============================================================

def load_alignment(path):
    pairs = []
    with open(path, "r", encoding="utf-8") as f:
        next(f)
        for line in f:
            a, b = line.split("\t")[:2]
            pairs.append((int(a), int(b)))
    return pairs


def stream_triples(path, keep=None, max_rows=None, seed_ents=None):
    triples, ents, rels, times = [], set(), set(), set()
    neigh = defaultdict(set) if seed_ents is not None else None
    with open(path, "r", encoding="utf-8") as f:
        next(f)
        for i, line in enumerate(f):
            if max_rows is not None and i >= max_rows:
                break
            p = line.split()
            if len(p) < 4:
                continue
            h, r, t, ts = int(p[0]), int(p[1]), int(p[2]), int(p[3])
            if seed_ents is not None:
                if h in seed_ents: neigh[h].add(t)
                if t in seed_ents: neigh[t].add(h)
            if keep is not None and (h not in keep or t not in keep):
                continue
            if keep is None and seed_ents is not None:
                continue
            triples.append((h, r, t, ts))
            ents.update((h, t)); rels.add(r); times.add(ts)
    return triples, ents, rels, times, neigh


all_pairs_raw = load_alignment(PAIR_DIR / "alignment_all.tsv")
align_a = {a for a, b in all_pairs_raw}
align_b = {b for a, b in all_pairs_raw}
print(f"pairs={len(all_pairs_raw):,}")

keep_a, keep_b = set(align_a), set(align_b)
if HOP_EXPANSION > 0:
    _, _, _, _, n1 = stream_triples(PAIR_DIR / f"quadruples_{lang_a}.tsv",
                                    max_rows=MAX_QUADS_EN, seed_ents=align_a)
    _, _, _, _, n2 = stream_triples(PAIR_DIR / f"quadruples_{lang_b}.tsv",
                                    max_rows=MAX_QUADS_OTHER, seed_ents=align_b)
    for e in align_a:
        keep_a.update(list(n1.get(e, ()))[:MAX_NEIGHBORS_PER_NODE])
    for e in align_b:
        keep_b.update(list(n2.get(e, ()))[:MAX_NEIGHBORS_PER_NODE])
    del n1, n2

print(f"keep {lang_a}={len(keep_a):,} {lang_b}={len(keep_b):,}")

triples1, e1, r1, t1, _ = stream_triples(PAIR_DIR / f"quadruples_{lang_a}.tsv", keep=keep_a, max_rows=MAX_QUADS_EN)
triples2, e2, r2, t2, _ = stream_triples(PAIR_DIR / f"quadruples_{lang_b}.tsv", keep=keep_b, max_rows=MAX_QUADS_OTHER)
print(f"triples {lang_a}={len(triples1):,} {lang_b}={len(triples2):,}")
gc.collect()


pairs=484,783
keep en=484,783 fr=484,783
triples en=5,524,081 fr=5,524,081


0

In [6]:
# ============================================================
# Remap + TREA inward graph
# ============================================================

def make_maps(s):
    ids = sorted(s)
    return {x: i for i, x in enumerate(ids)}


e1_map, e2_map = make_maps(e1), make_maps(e2)
r_map = make_maps(r1 | r2)
all_times = sorted(t1 | t2)
time_map = {t: i + 1 for i, t in enumerate(all_times)}
num_times = len(all_times) + 1
sorted_time_ids = list(range(1, num_times))

N1, N2 = len(e1_map), len(e2_map)
NUM_ENTITIES = N1 + N2
NUM_RELS_RAW = len(r_map)


def remap(triples, em, rm, tm, off=0):
    out = []
    for h, r, t, ts in triples:
        if h in em and t in em:
            tau = tm.get(ts, 0)
            out.append((em[h] + off, rm[r], em[t] + off, tau, tau))  # point: begin=end
    return out


tr1 = remap(triples1, e1_map, r_map, time_map, 0)
tr2 = remap(triples2, e2_map, r_map, time_map, N1)
del triples1, triples2, e1, e2
gc.collect()

valid = np.array([
    [e1_map[a], e2_map[b] + N1]
    for a, b in all_pairs_raw if a in e1_map and b in e2_map
], dtype=np.int64)
np.random.shuffle(valid)
if len(valid) < TRAIN_SEED:
    raise ValueError(f"pairs {len(valid)} < TRAIN_SEED")
train_pair = valid[:TRAIN_SEED].copy()
dev_pair = valid[TRAIN_SEED:].copy()


def build_inward(triples, num_rels):
    inward = defaultdict(list)
    for h, r, t, ts, te in triples:
        inward[t].append((h, r, ts))
        inward[h].append((t, r + num_rels, te))
    return inward


inward = build_inward(tr1 + tr2, NUM_RELS_RAW)
del tr1, tr2
gc.collect()

print(f"N1={N1:,} N2={N2:,} total={NUM_ENTITIES:,}")
print(f"rels={NUM_RELS_RAW} times={num_times} train={len(train_pair)} dev={len(dev_pair):,}")


N1=484,543 N2=484,543 total=969,086
rels=521 times=2026 train=200 dev=484,343


In [7]:
# ============================================================
# Eval - chunked CSLS (memory-safe; identical metric to before)
# ============================================================
# The previous version materialised the full |dev| x |dev| similarity
# matrix AND an int64 argsort of it on the CPU. For a few hundred thousand
# dev pairs that alone needs hundreds of GB. This version:
#   pass 1: row-chunks -> c1 (row top-k mean) and running per-column top-k -> c2
#   pass 2: row-chunks -> CSLS scores -> rank of the true match
# so peak memory is O(chunk * |dev|), never O(|dev|^2).
# rank = number of candidates scoring strictly higher than the true match.

@torch.no_grad()
def csls_hits(vec, test_pair, top_k=(1, 5, 10), csls_k=10, max_pairs=None, chunk=1024):
    pairs = np.asarray(test_pair)
    if max_pairs and len(pairs) > max_pairs:
        pairs = pairs[np.random.choice(len(pairs), max_pairs, False)]
    dev = DEVICE
    L = F.normalize(torch.from_numpy(np.ascontiguousarray(vec[pairs[:, 0]])).float().to(dev), dim=1, eps=1e-8)
    R = F.normalize(torch.from_numpy(np.ascontiguousarray(vec[pairs[:, 1]])).float().to(dev), dim=1, eps=1e-8)
    n = L.size(0)
    k = max(1, min(csls_k, n))

    c1 = torch.empty(n, device=dev)
    col_topk = torch.full((k, n), float("-inf"), device=dev)   # running top-k per column
    for s in range(0, n, chunk):
        sim = L[s:s + chunk] @ R.T                              # (C, n)
        c1[s:s + chunk] = sim.topk(k, dim=1).values.mean(1)
        cand = torch.cat([col_topk, sim.topk(min(k, sim.size(0)), dim=0).values], 0)
        col_topk = cand.topk(k, dim=0).values
    c2 = col_topk.mean(0)

    ranks = torch.empty(n, dtype=torch.long, device=dev)
    for s in range(0, n, chunk):
        sim = 2 * (L[s:s + chunk] @ R.T) - c1[s:s + chunk, None] - c2[None, :]
        true = sim[torch.arange(sim.size(0), device=dev), torch.arange(s, s + sim.size(0), device=dev)]
        ranks[s:s + chunk] = (sim > true[:, None]).sum(1)

    ranks = ranks.float().cpu().numpy()
    print(f"CSLS n={n}")
    for kk in top_k:
        print(f"  Hits@{kk}: {(ranks < kk).mean() * 100:.2f}%")
    print(f"  MRR: {(1.0 / (ranks + 1)).mean():.3f}")
    del L, R
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

print("eval ready (chunked CSLS)")


eval ready (chunked CSLS)


In [8]:
# ============================================================
# TREA model - MEMORY-SAFE version (same maths, same parameter names)
# ============================================================
# WHY THE PREVIOUS VERSION RAN OUT OF GPU MEMORY
#   TRALayer built `src_feat = h[src_idx]` and then
#   `alpha*(Wt*src_feat) + beta*(Wr*src_feat)` -> several tensors of shape
#   (E, 3*DIM) per layer, and autograd keeps most of them for backward.
#   With E ~ 4.2M inward links and 3*DIM = 300 that is ~4.7 GiB PER TENSOR.
#
# WHAT CHANGED (output is mathematically identical):
#   1. Wt / Wr are diagonal vectors shared by every edge, so they factor
#      out of the neighbour sum:
#          out_i = Wt * sum_e alpha_e h[src_e]  +  Wr * sum_e beta_e h[src_e]
#      The two weighted sums are computed in EDGE CHUNKS by a custom
#      autograd function whose backward is also chunked. Nothing of shape
#      (E, F) is ever stored - peak extra memory is O(CHUNK * F).
#   2. Attention scores are computed per relation / per timestamp and then
#      indexed:  (rel_emb @ v_r)[rel_ids]  ==  rel_emb[rel_ids] @ v_r
#      which avoids two (E, DIM) gathers per layer.
#   3. NAR mean-aggregation uses the same chunked scatter, and the relation
#      / time means are computed once and reused (the original computed the
#      identical quantity twice).
#   4. Optional activation checkpointing per attention layer.
# Parameter and buffer names are unchanged, so old checkpoints still load.

from torch.utils.checkpoint import checkpoint as _ckpt


class _ChunkedScatter(torch.autograd.Function):
    """out[row[e]] += table[col[e]]  for all e, in chunks (no (E,F) tensor)."""

    @staticmethod
    def forward(ctx, table, col, row, n, chunk):
        out = torch.zeros(n, table.shape[1], device=table.device, dtype=table.dtype)
        for s in range(0, col.numel(), chunk):
            out.index_add_(0, row[s:s + chunk], table[col[s:s + chunk]])
        ctx.save_for_backward(col, row)
        ctx.tbl_shape, ctx.chunk = tuple(table.shape), chunk
        return out

    @staticmethod
    def backward(ctx, g):
        col, row = ctx.saved_tensors
        gt = torch.zeros(ctx.tbl_shape, device=g.device, dtype=g.dtype)
        for s in range(0, col.numel(), ctx.chunk):
            gt.index_add_(0, col[s:s + ctx.chunk], g[row[s:s + ctx.chunk]])
        return gt, None, None, None, None


class _ChunkedWeightedAgg2(torch.autograd.Function):
    """
    A[i] = sum_{e: dst_e=i} alpha_e * h[src_e]
    B[i] = sum_{e: dst_e=i} beta_e  * h[src_e]
    computed (and back-propagated) in edge chunks.
    """

    @staticmethod
    def forward(ctx, h, alpha, beta, src, dst, n, chunk):
        A = torch.zeros(n, h.shape[1], device=h.device, dtype=h.dtype)
        B = torch.zeros(n, h.shape[1], device=h.device, dtype=h.dtype)
        for s in range(0, src.numel(), chunk):
            sl = slice(s, s + chunk)
            hs = h[src[sl]]
            A.index_add_(0, dst[sl], alpha[sl].unsqueeze(-1) * hs)
            B.index_add_(0, dst[sl], beta[sl].unsqueeze(-1) * hs)
        ctx.save_for_backward(h, alpha, beta, src, dst)
        ctx.chunk = chunk
        return A, B

    @staticmethod
    def backward(ctx, gA, gB):
        h, alpha, beta, src, dst = ctx.saved_tensors
        gh = torch.zeros_like(h)
        galpha = torch.zeros_like(alpha)
        gbeta = torch.zeros_like(beta)
        for s in range(0, src.numel(), ctx.chunk):
            sl = slice(s, s + ctx.chunk)
            hs = h[src[sl]]
            ga = gA[dst[sl]]
            gb = gB[dst[sl]]
            galpha[sl] = (ga * hs).sum(-1)
            gbeta[sl] = (gb * hs).sum(-1)
            gh.index_add_(0, src[sl],
                          alpha[sl].unsqueeze(-1) * ga + beta[sl].unsqueeze(-1) * gb)
        return gh, galpha, gbeta, None, None, None, None


def scatter_mean(row, col, table, n, chunk):
    out = _ChunkedScatter.apply(table, col, row, n, chunk)
    cnt = torch.zeros(n, device=table.device, dtype=table.dtype)
    cnt.index_add_(0, row, torch.ones(row.numel(), device=table.device, dtype=table.dtype))
    return out / cnt.clamp_min(1.0).unsqueeze(-1)


class NAR(nn.Module):
    def __init__(self, ent_emb, rel_emb, time_emb, inward, num_entities, chunk=262144):
        super().__init__()
        self.ent_emb, self.rel_emb, self.time_emb = ent_emb, rel_emb, time_emb
        self.num_entities = num_entities
        self.chunk = chunk
        ent_rows, ent_cols, rel_rows, rel_cols, time_rows, time_cols = [], [], [], [], [], []
        for e in range(num_entities):
            links = inward.get(e, [])
            ent_rows.append(e); ent_cols.append(e)
            for src, r, t in links:
                ent_rows.append(e); ent_cols.append(src)
                rel_rows.append(e); rel_cols.append(r)
                time_rows.append(e); time_cols.append(t)
        self.register_buffer("ent_row", torch.tensor(ent_rows, dtype=torch.long))
        self.register_buffer("ent_col", torch.tensor(ent_cols, dtype=torch.long))
        self.register_buffer("rel_row", torch.tensor(rel_rows, dtype=torch.long))
        self.register_buffer("rel_col", torch.tensor(rel_cols, dtype=torch.long))
        self.register_buffer("time_row", torch.tensor(time_rows, dtype=torch.long))
        self.register_buffer("time_col", torch.tensor(time_cols, dtype=torch.long))

    def parts(self):
        n, c = self.num_entities, self.chunk
        e = scatter_mean(self.ent_row, self.ent_col, self.ent_emb.weight, n, c)
        r = scatter_mean(self.rel_row, self.rel_col, self.rel_emb.weight, n, c)
        t = scatter_mean(self.time_row, self.time_col, self.time_emb.weight, n, c)
        return e, r, t

    def forward(self):
        e, r, t = self.parts()
        return torch.cat([e, r, t], dim=-1)


class TRALayer(nn.Module):
    def __init__(self, feat_dim, time_rel_dim, chunk=262144):
        super().__init__()
        self.chunk = chunk
        self.v_t = nn.Parameter(torch.empty(time_rel_dim))
        self.v_r = nn.Parameter(torch.empty(time_rel_dim))
        self.Wt = nn.Parameter(torch.empty(feat_dim))
        self.Wr = nn.Parameter(torch.empty(feat_dim))
        for p in (self.v_t, self.v_r, self.Wt, self.Wr):
            nn.init.normal_(p, std=0.1)

    @staticmethod
    def _segment_softmax(scores, seg_ids, num_segments):
        seg_max = torch.full((num_segments,), float("-inf"), device=scores.device, dtype=scores.dtype)
        seg_max.scatter_reduce_(0, seg_ids, scores, reduce="amax", include_self=True)
        seg_max = torch.nan_to_num(seg_max, neginf=0.0)
        exp_s = torch.exp(scores - seg_max[seg_ids])
        seg_sum = torch.zeros(num_segments, device=scores.device, dtype=scores.dtype)
        seg_sum.scatter_add_(0, seg_ids, exp_s)
        return exp_s / seg_sum[seg_ids].clamp_min(1e-16)

    def forward(self, h, src_idx, dst_idx, rel_ids, time_ids, rel_emb, time_emb, num_entities):
        # score per relation / per timestamp, then index -> (E,) not (E, DIM)
        alpha = self._segment_softmax((time_emb @ self.v_t)[time_ids], dst_idx, num_entities)
        beta = self._segment_softmax((rel_emb @ self.v_r)[rel_ids], dst_idx, num_entities)
        A, B = _ChunkedWeightedAgg2.apply(h, alpha, beta, src_idx, dst_idx, num_entities, self.chunk)
        return torch.tanh(self.Wt * A + self.Wr * B)


class TREA(nn.Module):
    def __init__(self, num_entities, num_rels_doubled, num_times, dim, depth, inward,
                 chunk=262144, use_checkpoint=True):
        super().__init__()
        self.num_entities = num_entities
        self.chunk = chunk
        self.use_checkpoint = use_checkpoint
        self.ent_emb = nn.Embedding(num_entities, dim)
        self.rel_emb = nn.Embedding(num_rels_doubled, dim)
        self.time_emb = nn.Embedding(num_times, dim)
        nn.init.xavier_uniform_(self.ent_emb.weight)
        nn.init.xavier_uniform_(self.rel_emb.weight)
        nn.init.xavier_uniform_(self.time_emb.weight)
        self.nar = NAR(self.ent_emb, self.rel_emb, self.time_emb, inward, num_entities, chunk)
        feat_dim = 3 * dim
        self.layers = nn.ModuleList([TRALayer(feat_dim, dim, chunk) for _ in range(depth)])

        src_list, dst_list, rel_list, time_list = [], [], [], []
        for e in range(num_entities):
            for src, r, t in inward.get(e, []):
                src_list.append(src); dst_list.append(e)
                rel_list.append(r); time_list.append(t)
        if not src_list:
            src_list = [0]; dst_list = [0]; rel_list = [0]; time_list = [0]
        self.register_buffer("src_idx", torch.tensor(src_list, dtype=torch.long))
        self.register_buffer("dst_idx", torch.tensor(dst_list, dtype=torch.long))
        self.register_buffer("rel_ids", torch.tensor(rel_list, dtype=torch.long))
        self.register_buffer("time_ids", torch.tensor(time_list, dtype=torch.long))

    def forward(self):
        e, r, t = self.nar.parts()           # r, t double as the "neighbour mean" of Eq. 6
        h = torch.cat([e, r, t], dim=-1)
        outputs = [h]
        use_ckpt = self.use_checkpoint and self.training and torch.is_grad_enabled()
        for layer in self.layers:
            args = (h, self.src_idx, self.dst_idx, self.rel_ids, self.time_ids,
                    self.rel_emb.weight, self.time_emb.weight, self.num_entities)
            h = _ckpt(layer, *args, use_reentrant=False) if use_ckpt else layer(*args)
            outputs.append(h)
        return torch.cat(outputs + [r, t], dim=-1)


def mml_loss_full(emb, pos_pairs, gamma):
    ei = emb[pos_pairs[:, 0]]; ej = emb[pos_pairs[:, 1]]
    d_pos = ((ei - ej) ** 2).sum(-1)
    emb_sq = (emb ** 2).sum(-1)
    ei_sq = (ei ** 2).sum(-1, keepdim=True)
    d_all_j = ei_sq + emb_sq.unsqueeze(0) - 2 * (ei @ emb.t())
    term1 = torch.logsumexp(torch.cat([torch.zeros_like(d_pos).unsqueeze(1),
                                       gamma + d_pos.unsqueeze(1) - d_all_j], 1), 1)
    ej_sq = (ej ** 2).sum(-1, keepdim=True)
    d_all_i = ej_sq + emb_sq.unsqueeze(0) - 2 * (ej @ emb.t())
    term2 = torch.logsumexp(torch.cat([torch.zeros_like(d_pos).unsqueeze(1),
                                       gamma + d_pos.unsqueeze(1) - d_all_i], 1), 1)
    return (term1 + term2).mean()


def mml_loss_sampled(emb, pos_pairs, gamma, num_neg, n1, n2):
    """Sampled negatives - scales to large |V|."""
    device = emb.device
    ei = emb[pos_pairs[:, 0]]; ej = emb[pos_pairs[:, 1]]
    d_pos = ((ei - ej) ** 2).sum(-1)
    B = pos_pairs.size(0)
    neg_i = torch.randint(0, n1, (B, num_neg), device=device)
    neg_j = torch.randint(n1, n1 + n2, (B, num_neg), device=device)
    d_neg_j = ((ei.unsqueeze(1) - emb[neg_j]) ** 2).sum(-1)
    d_neg_i = ((emb[neg_i] - ej.unsqueeze(1)) ** 2).sum(-1)
    term1 = torch.logsumexp(torch.cat([
        torch.zeros(B, 1, device=device),
        gamma + d_pos.unsqueeze(1) - d_neg_j
    ], 1), 1)
    term2 = torch.logsumexp(torch.cat([
        torch.zeros(B, 1, device=device),
        gamma + d_pos.unsqueeze(1) - d_neg_i
    ], 1), 1)
    return (term1 + term2).mean()


def sequential_time_regularizer(time_emb, sorted_time_ids, h_tb):
    ids = torch.tensor(sorted_time_ids, dtype=torch.long, device=time_emb.device)
    if len(ids) < 2:
        return time_emb.sum() * 0.0
    h_seq = time_emb[ids]
    return ((h_seq[1:] - h_seq[:-1] - h_tb) ** 2).sum(-1).mean()

print("TREA (memory-safe) defined")


TREA (memory-safe) defined


In [9]:
# ============================================================
# Build model
# ============================================================

model = TREA(NUM_ENTITIES, NUM_RELS_RAW * 2, num_times, DIM, DEPTH, inward,
             chunk=CHUNK_EDGES, use_checkpoint=USE_CHECKPOINT).to(DEVICE)
h_tb = nn.Parameter(torch.zeros(DIM, device=DEVICE))
optimizer = torch.optim.RMSprop(list(model.parameters()) + [h_tb], lr=LR)
train_pair_t = torch.tensor(train_pair, dtype=torch.long, device=DEVICE)

USE_FULL_MML = NUM_ENTITIES <= FULL_MML_MAX_ENTITIES
E_LINKS = model.src_idx.numel()
print(f"Entities={NUM_ENTITIES:,}  inward links E={E_LINKS:,}  full_MML={USE_FULL_MML}")
print(f"Old per-edge tensor would have been ~{E_LINKS*3*DIM*4/2**30:.2f} GiB EACH (now never allocated)")
print("Params:", sum(p.numel() for p in model.parameters()))
if torch.cuda.is_available():
    print(f"GPU alloc {torch.cuda.memory_allocated()/1e9:.2f} GB")


Entities=969,086  inward links E=22,096,324  full_MML=False
Old per-edge tensor would have been ~24.69 GiB EACH (now never allocated)
Params: 97217000
GPU alloc 2.17 GB


In [10]:
# ============================================================
# Train
# ============================================================

if torch.cuda.is_available():
    torch.cuda.reset_peak_memory_stats()

for epoch in range(EPOCHS):
    model.train()
    optimizer.zero_grad()
    emb = model()
    if USE_FULL_MML:
        loss_p = mml_loss_full(emb, train_pair_t, GAMMA)
    else:
        loss_p = mml_loss_sampled(emb, train_pair_t, GAMMA, NUM_NEG_SAMPLES, N1, N2)
    loss_str = sequential_time_regularizer(model.time_emb.weight, sorted_time_ids, h_tb)
    loss = loss_p + LAMBDA_T * loss_str
    loss.backward()
    optimizer.step()

    if epoch == 0 and torch.cuda.is_available():
        print(f"[epoch 001] peak GPU memory in a training step: "
              f"{torch.cuda.max_memory_allocated()/2**30:.2f} GiB "
              f"of {torch.cuda.get_device_properties(0).total_memory/2**30:.2f} GiB")

    if epoch % 10 == 9 or epoch == EPOCHS - 1:
        model.eval()
        with torch.no_grad():
            emb_np = model().detach().cpu().numpy()
        print(f"[epoch {epoch+1:03d}] loss={loss.item():.4f} "
              f"(MML={loss_p.item():.4f} STR={loss_str.item():.4f})")
        csls_hits(emb_np, dev_pair, TOP_K, CSLS_K, EVAL_MAX_PAIRS)
        del emb_np
        if torch.cuda.is_available():
            torch.cuda.empty_cache()

print("Training complete.")


OutOfMemoryError: CUDA out of memory. Tried to allocate 3.97 GiB. GPU 0 has a total capacity of 14.56 GiB of which 2.19 GiB is free. Including non-PyTorch memory, this process has 12.38 GiB memory in use. Of the allocated memory 12.21 GiB is allocated by PyTorch, and 42.95 MiB is reserved by PyTorch but unallocated. If reserved but unallocated memory is large try setting PYTORCH_CUDA_ALLOC_CONF=expandable_segments:True to avoid fragmentation.  See documentation for Memory Management  (https://docs.pytorch.org/docs/stable/notes/cuda.html#optimizing-memory-usage-with-pytorch-cuda-alloc-conf)

In [ ]:
# ============================================================
# Final eval + save
# ============================================================

model.eval()
with torch.no_grad():
    final = model().detach().cpu().numpy()

print("shape", final.shape)
csls_hits(final, dev_pair, TOP_K, CSLS_K, max_pairs=None)

SAVE = f"/content/drive/MyDrive/trea_tkgl_{LANG_PAIR}_checkpoint.pt"
torch.save({
    "model": model.state_dict(),
    "h_tb": h_tb.detach().cpu(),
    "config": {
        "LANG_PAIR": LANG_PAIR, "HOP_EXPANSION": HOP_EXPANSION,
        "TRAIN_SEED": TRAIN_SEED, "DIM": DIM, "DEPTH": DEPTH,
        "N1": N1, "N2": N2, "USE_FULL_MML": USE_FULL_MML,
    },
}, SAVE)
print("Saved", SAVE)


## Notes

- Default `HOP_EXPANSION=0` → only alignment entities (fits Colab T4).
- If `N > 30_000`, **sampled MML** is used automatically.
- Switch pair: `LANG_PAIR` ∈ `{en_fa, en_zh, en_ja, en_fr}`.
- Still OOM? First check the "peak GPU memory" line printed at epoch 1. Then, in order:
  `CHUNK_EDGES=65_536` -> `DIM=64` -> `DEPTH=1` -> `MAX_QUADS_EN=200000`.
- Memory that still scales with the number of ENTITIES (not edges) is `N x 3*DIM` activations; `USE_CHECKPOINT=True` keeps that small.
- Document subgraph + loss variant in your thesis experimental settings.
